In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework_06") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 18:56:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [8]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
df.repartition(4).write.mode("overwrite").parquet('data/pq/yellow/2025/11/')
print("Repartitioning complete!")

[Stage 6:>                                                          (0 + 4) / 4]

Repartitioning complete!


In [6]:
ls -lh data/pq/yellow/2025/11/

total 205312
-rw-r--r--@ 1 ying  staff     0B Mar  4 19:01 _SUCCESS
-rw-r--r--@ 1 ying  staff    24M Mar  4 19:01 part-00000-53ab7d33-721c-4545-bf94-d488f3e71db4-c000.snappy.parquet
-rw-r--r--@ 1 ying  staff    24M Mar  4 19:01 part-00001-53ab7d33-721c-4545-bf94-d488f3e71db4-c000.snappy.parquet
-rw-r--r--@ 1 ying  staff    24M Mar  4 19:01 part-00002-53ab7d33-721c-4545-bf94-d488f3e71db4-c000.snappy.parquet
-rw-r--r--@ 1 ying  staff    24M Mar  4 19:01 part-00003-53ab7d33-721c-4545-bf94-d488f3e71db4-c000.snappy.parquet


In [10]:
from pyspark.sql import functions as F
df.filter(F.to_date(df.tpep_pickup_datetime) == '2025-11-15').count()

162604

In [11]:
df_duration = df.withColumn('duration_hours', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
)

df_duration.select(F.max('duration_hours')).show()

+-------------------+
|max(duration_hours)|
+-------------------+
|  90.64666666666666|
+-------------------+



In [16]:
!curl -O https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12331  100 12331    0     0   185k      0 --:--:-- --:--:-- --:--:--  188k


In [17]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [18]:
df.createOrReplaceTempView('yellow_taxi_data')
df_zones.createOrReplaceTempView('zones')

In [19]:
spark.sql("""
    SELECT 
        z.Zone, 
        COUNT(1) as pickup_count
    FROM 
        yellow_taxi_data t
    JOIN 
        zones z ON t.PULocationID = z.LocationID
    GROUP BY 
        1
    ORDER BY 
        pickup_count ASC
    LIMIT 5
""").show()

[Stage 18:>                                                         (0 + 4) / 4]

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
|Eltingville/Annad...|           1|
|       Arden Heights|           1|
|       Port Richmond|           3|
|       Rikers Island|           4|
+--------------------+------------+

